In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge/sample_submission.csv
/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge/classes.txt
/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge/train_labels.csv
/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge/train_images/024458.png
/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge/train_images/026593.png
/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge/train_images/020735.png
/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge/train_images/016018.png
/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge/train_images/024532.png
/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge/train_images/014427.png
/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge/train

In [2]:
import os
import gc
import math
import copy
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torchvision import transforms

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# =============================================================================
# CONFIG — tuned for single T4, 9h limit
# =============================================================================
DATA_DIR  = '/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge'
TRAIN_DIR = os.path.join(DATA_DIR, 'train_images')
TEST_DIR  = os.path.join(DATA_DIR, 'test_images')
CKPT_DIR  = '/kaggle/working'
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CLASSES   = ['airplane','automobile','bird','cat','deer',
             'dog','frog','horse','ship','truck']
C2I = {c: i for i, c in enumerate(CLASSES)}
I2C = {i: c for i, c in enumerate(CLASSES)}

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

CFG = {
    # Model
    'depth'        : 28,
    'widen'        : 10,
    'dropout'      : 0.3,

    # Training — THE FIX: large batch is mandatory with lr=0.1 for SGD
    # Linear scaling rule: lr / batch_size = constant
    # Top solution: lr=0.1, batch=512  → ratio = 0.000195
    # If you use batch=128: lr must be 0.025, NOT 0.1
    'batch_size'   : 512,
    'epochs'       : 300,
    'lr'           : 0.1,
    'momentum'     : 0.9,
    'weight_decay' : 5e-4,
    'warmup_epochs': 5,
    'label_smooth' : 0.1,

    # SWA: start at 80% of training (same ratio as top solution: 360/450)
    'swa_start'    : 240,   # 240/300 = 80%
    'swa_lr'       : 0.005,

    # MixUp / CutMix
    'mix_prob'     : 0.5,
    'mix_alpha'    : 1.0,

    # Inference
    'tta_views'    : 30,
    'tta_batch'    : 256,

    # Data
    'val_frac'     : 0.05,
    'seed'         : 42,
    'num_workers'  : 4,

    # Multi-seed ensemble (set to [42, 137] for 2-seed run if time allows)
    'seeds'        : [42],
}

# =============================================================================
# SEED
# =============================================================================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =============================================================================
# DATASET
# =============================================================================
class ShiftGuardDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.is_test   = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img_id = str(row['id']).zfill(6)
        path   = os.path.join(self.img_dir, f'{img_id}.png')
        if not os.path.exists(path):
            path = os.path.join(self.img_dir, f'{img_id}.jpg')

        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)

        if self.is_test:
            return img, str(row['id']).zfill(6)
        return img, C2I[row['label']]

class Cutout:
   
    def __init__(self, length=16):
        self.length = length
    def __call__(self, img_tensor):
        h, w = img_tensor.size(1), img_tensor.size(2)
        y = random.randint(0, h - 1)
        x = random.randint(0, w - 1)
        y1 = max(0, y - self.length // 2)
        y2 = min(h, y + self.length // 2)
        x1 = max(0, x - self.length // 2)
        x2 = min(w, x + self.length // 2)
        img_tensor[:, y1:y2, x1:x2] = 0.0
        return img_tensor

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4, fill=128),
    transforms.RandomHorizontalFlip(),
    
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    Cutout(length=16),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# TTA: crop + flip + colour + ROTATION

tta_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4, fill=128),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.RandomRotation(degrees=10, fill=128),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])




class BalancedSoftmaxLoss(nn.Module):
    def __init__(self, class_counts, label_smoothing=0.1):
        super().__init__()
        freq = torch.tensor(class_counts, dtype=torch.float32)
        freq = freq / freq.sum()
        # We add log(freq) to each logit — effectively an additive bias
        self.register_buffer('log_freq', torch.log(freq + 1e-12))
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        # Shift logits by log(class_frequency) before computing CE
        adjusted = logits + self.log_freq.unsqueeze(0)
        return F.cross_entropy(adjusted, targets,
                               label_smoothing=self.label_smoothing)


# =============================================================================
# WEIGHTED SAMPLER — sqrt-inverse (same as top solution)
# Combines with BalancedSoftmaxLoss: sampler ensures tail classes are SEEN
# enough, loss ensures the decision boundaries are geometrically correct.
# =============================================================================

def make_sampler(df):
    counts  = np.bincount([C2I[l] for l in df['label']], minlength=10)
    # sqrt-inverse: less aggressive than full inverse, avoids overfitting tails
    w_class = 1.0 / (np.sqrt(counts) + 1e-6)
    weights = [w_class[C2I[df.iloc[i]['label']]] for i in range(len(df))]
    return WeightedRandomSampler(weights, len(df), replacement=True)


# =============================================================================
# MODEL: WideResNet-28-10 with Kaiming He init
# The init is important for WRN-28-10 because of its depth.
# Default PyTorch init (uniform/normal) can lead to vanishing/exploding
# activations in the first few epochs, causing the loss to plateau early.
# =============================================================================

class WideBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride, dropout):
        super().__init__()
        self.bn1   = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride,
                               padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1,
                               padding=1, bias=False)
        self.drop  = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()
        self.skip  = (nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)
                      if stride != 1 or in_ch != out_ch else nn.Identity())

    def forward(self, x):
        out = self.drop(self.conv1(F.relu(self.bn1(x), inplace=True)))
        out = self.conv2(F.relu(self.bn2(out), inplace=True))
        return out + self.skip(x)


class WideResNet(nn.Module):
    """WideResNet-28-10. ~36.5M params. Native CIFAR-resolution architecture."""
    def __init__(self, depth=28, widen=10, dropout=0.3, num_classes=10):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n  = (depth - 4) // 6
        ch = [16, 16 * widen, 32 * widen, 64 * widen]

        self.conv1  = nn.Conv2d(3, ch[0], 3, stride=1, padding=1, bias=False)
        self.group1 = self._make(ch[0], ch[1], n, 1, dropout)
        self.group2 = self._make(ch[1], ch[2], n, 2, dropout)
        self.group3 = self._make(ch[2], ch[3], n, 2, dropout)
        self.bn     = nn.BatchNorm2d(ch[3])
        self.fc     = nn.Linear(ch[3], num_classes)

        self._init_weights()

    def _make(self, ic, oc, n, stride, dropout):
        layers = [WideBlock(ic, oc, stride, dropout)]
        for _ in range(1, n):
            layers.append(WideBlock(oc, oc, 1, dropout))
        return nn.Sequential(*layers)

    def _init_weights(self):
        """Kaiming He init for conv, standard init for BN."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out',
                                        nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        out = self.conv1(x)
        out = self.group1(out)
        out = self.group2(out)
        out = self.group3(out)
        out = F.relu(self.bn(out), inplace=True)
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.fc(out)


# =============================================================================
# MIXUP / CUTMIX
# =============================================================================

def rand_bbox(H, W, lam):
    cut_ratio = math.sqrt(1.0 - lam)
    cut_h, cut_w = int(H * cut_ratio), int(W * cut_ratio)
    cy, cx = random.randint(0, H - 1), random.randint(0, W - 1)
    y1 = max(0, cy - cut_h // 2); y2 = min(H, cy + cut_h // 2)
    x1 = max(0, cx - cut_w // 2); x2 = min(W, cx + cut_w // 2)
    return y1, y2, x1, x2

def mixup(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def cutmix(x, y, alpha=1.0):
    lam  = np.random.beta(alpha, alpha)
    idx  = torch.randperm(x.size(0), device=x.device)
    H, W = x.size(2), x.size(3)
    y1, y2, x1, x2 = rand_bbox(H, W, lam)
    x_out = x.clone()
    x_out[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1.0 - (y2 - y1) * (x2 - x1) / (H * W)
    return x_out, y, y[idx], lam

def mix_loss(criterion, logits, ya, yb, lam):
    return lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)


# =============================================================================
# LEARNING RATE SCHEDULE
# 5-epoch linear warmup → cosine decay to SWA start
# After SWA start: SWALR takes over (constant low LR for weight averaging)
# =============================================================================

def make_lr_lambda(warmup, swa_start, total):
    def fn(epoch):
        if epoch < warmup:
            return (epoch + 1) / warmup
        progress = (epoch - warmup) / max(1, swa_start - warmup)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0))))
    return fn


# =============================================================================
# TRAINING LOOP
# =============================================================================

def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = 0.0

    for imgs, labels in loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # MixUp or CutMix with probability mix_prob
        if random.random() < CFG['mix_prob']:
            if random.random() < 0.5:
                imgs, ya, yb, lam = mixup(imgs, labels, CFG['mix_alpha'])
            else:
                imgs, ya, yb, lam = cutmix(imgs, labels, CFG['mix_alpha'])
            mixed = True
        else:
            ya, yb, lam, mixed = labels, labels, 1.0, False

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            loss   = mix_loss(criterion, logits, ya, yb, lam) if mixed \
                     else criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    preds, trues = [], []
    total_loss   = 0.0

    for imgs, labels in loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item()
        preds.extend(logits.argmax(1).cpu().numpy())
        trues.extend(labels.cpu().numpy())

    f1 = f1_score(trues, preds, average='macro', zero_division=0)
    return total_loss / len(loader), f1, preds, trues


# =============================================================================
# INFERENCE WITH 30-VIEW TTA
# Averages: 1 clean view + 29 stochastic augmented views
# Each view is an independent random draw from tta_transform
# =============================================================================

@torch.no_grad()
def predict_tta(model, test_df, device, n_views=30):
    model.eval()

    # --- Clean pass ---
    ds     = ShiftGuardDataset(test_df, TEST_DIR,
                               transform=val_transform, is_test=True)
    loader = DataLoader(ds, batch_size=CFG['tta_batch'], shuffle=False,
                        num_workers=CFG['num_workers'], pin_memory=True)

    all_logits = []
    all_ids    = []
    with torch.amp.autocast('cuda'):
        for imgs, ids in loader:
            all_logits.append(model(imgs.to(device)).cpu())
            all_ids.extend(ids)
    acc = torch.cat(all_logits, dim=0)  # [N, 10]
    print(f'  Clean view done ({len(all_ids)} samples)')

    # --- Stochastic TTA passes ---
    ds_tta = ShiftGuardDataset(test_df, TEST_DIR,
                               transform=tta_transform, is_test=True)
    loader_tta = DataLoader(ds_tta, batch_size=CFG['tta_batch'], shuffle=False,
                            num_workers=CFG['num_workers'], pin_memory=True)
    for v in range(n_views - 1):
        buf = []
        with torch.amp.autocast('cuda'):
            for imgs, _ in loader_tta:
                buf.append(model(imgs.to(device)).cpu())
        acc = acc + torch.cat(buf, dim=0)
        if (v + 1) % 10 == 0 or (v + 1) == n_views - 1:
            print(f'  TTA {v+2}/{n_views}')

    # Average logits across all views then argmax
    return acc / n_views, all_ids


# =============================================================================
# MAIN
# =============================================================================

def train_one_seed(seed, df, class_counts):
    print(f'\n{"="*60}')
    print(f'  Training seed={seed}  |  epochs={CFG["epochs"]}')
    print(f'  batch={CFG["batch_size"]}  lr={CFG["lr"]}  SWA from ep{CFG["swa_start"]}')
    print(f'{"="*60}')

    seed_everything(seed)

    train_df, val_df = train_test_split(
        df, test_size=CFG['val_frac'], stratify=df['label'],
        random_state=CFG['seed']
    )

    sampler = make_sampler(train_df)
    train_ds = ShiftGuardDataset(train_df, TRAIN_DIR, transform=train_transform)
    val_ds   = ShiftGuardDataset(val_df,   TRAIN_DIR, transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                              sampler=sampler,
                              num_workers=CFG['num_workers'],
                              pin_memory=True, persistent_workers=True,
                              drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=CFG['batch_size'],
                              shuffle=False,
                              num_workers=CFG['num_workers'],
                              pin_memory=True, persistent_workers=True)

    # Build model
    model = WideResNet(depth=CFG['depth'], widen=CFG['widen'],
                       dropout=CFG['dropout']).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  WRN-{CFG["depth"]}-{CFG["widen"]} | {n_params/1e6:.1f}M params')

    # BalancedSoftmaxLoss — the main mathematical improvement over your code
    criterion = BalancedSoftmaxLoss(class_counts,
                                    label_smoothing=CFG['label_smooth']).to(DEVICE)

    optimizer = optim.SGD(model.parameters(), lr=CFG['lr'],
                          momentum=CFG['momentum'],
                          weight_decay=CFG['weight_decay'],
                          nesterov=True)

    swa_model     = AveragedModel(model)
    cosine_sched  = optim.lr_scheduler.LambdaLR(
        optimizer,
        make_lr_lambda(CFG['warmup_epochs'], CFG['swa_start'], CFG['epochs'])
    )
    swa_sched     = SWALR(optimizer, swa_lr=CFG['swa_lr'],
                          anneal_epochs=10, anneal_strategy='cos')
    scaler        = torch.amp.GradScaler('cuda')

    best_f1    = 0.0
    best_state = None
    in_swa     = False

    for epoch in range(CFG['epochs']):
        # Switch scheduler at SWA boundary
        if epoch == CFG['swa_start'] and not in_swa:
            in_swa = True
            print(f'\n  >>> Switching to SWA at epoch {epoch+1}')

        loss = train_epoch(model, train_loader, criterion, optimizer,
                           scaler, DEVICE)

        if in_swa:
            swa_model.update_parameters(model)
            swa_sched.step()
        else:
            cosine_sched.step()

        # Validate every 5 epochs
        if (epoch + 1) % 5 == 0 or (epoch + 1) == CFG['epochs']:
            eval_model = swa_model if in_swa else model
            _, val_f1, preds, trues = validate(eval_model, val_loader,
                                               criterion, DEVICE)
            lr_now = optimizer.param_groups[0]['lr']
            tag = ' [SWA]' if in_swa else ''
            print(f'  Ep {epoch+1:3d}/{CFG["epochs"]} | '
                  f'loss {loss:.4f} | val F1 {val_f1:.4f} | '
                  f'lr {lr_now:.5f}{tag}')

            if val_f1 > best_f1:
                best_f1    = val_f1
                best_state = copy.deepcopy(
                    swa_model.module.state_dict() if in_swa
                    else model.state_dict()
                )
                ckpt = os.path.join(CKPT_DIR, f'best_seed{seed}.pth')
                torch.save(best_state, ckpt)
                print(f'  -> New best F1 {best_f1:.4f} saved to {ckpt}')

    # Finalize SWA: update BN statistics using training data
    print('\n  Updating SWA batch normalization stats...')
    bn_ds     = ShiftGuardDataset(train_df, TRAIN_DIR, transform=train_transform)
    bn_loader = DataLoader(bn_ds, batch_size=256, shuffle=True,
                           num_workers=CFG['num_workers'])
    update_bn(bn_loader, swa_model, device=DEVICE)

    _, swa_f1, swa_preds, swa_trues = validate(
        swa_model, val_loader, criterion, DEVICE
    )
    print(f'  SWA final val F1: {swa_f1:.4f}')
    print(classification_report(swa_trues, swa_preds,
                                 target_names=CLASSES, zero_division=0))

    # Use SWA weights if they beat the best checkpoint
    if swa_f1 > best_f1:
        best_f1    = swa_f1
        best_state = copy.deepcopy(swa_model.module.state_dict())
        print(f'  SWA beats checkpoint — using SWA weights')

    swa_ckpt = os.path.join(CKPT_DIR, f'swa_seed{seed}.pth')
    torch.save(swa_model.module.state_dict(), swa_ckpt)
    print(f'  SWA weights saved: {swa_ckpt}')
    print(f'  Seed {seed} best F1: {best_f1:.4f}')

    del model, swa_model, optimizer, scaler
    gc.collect()
    torch.cuda.empty_cache()

    return best_state, best_f1


def main():
    print(f'Device: {DEVICE}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')

    # Load data
    df = pd.read_csv(os.path.join(DATA_DIR, 'train_labels.csv'),
                     dtype={'id': str})
    df['id'] = df['id'].str.zfill(6)

    # Compute class counts from full training split (95%)
    train_df_full, _ = train_test_split(
        df, test_size=CFG['val_frac'], stratify=df['label'],
        random_state=CFG['seed']
    )
    class_counts = np.bincount(
        [C2I[l] for l in train_df_full['label']], minlength=10
    )
    print(f'Training samples: {len(train_df_full)}')
    print('Class distribution:')
    for cls, cnt in zip(CLASSES, class_counts):
        print(f'  {cls:12s}: {cnt}')

    # Train each seed
    all_states = []
    for seed in CFG['seeds']:
        state, f1 = train_one_seed(seed, df, class_counts)
        all_states.append(state)

    # =========================================================================
    # ENSEMBLE INFERENCE: 30-view TTA, logit-level average across all seeds
    # =========================================================================
    print(f'\n{"="*60}')
    print(f'  Inference: {len(all_states)} model(s) × {CFG["tta_views"]} TTA views')
    print(f'{"="*60}')

    test_df  = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'),
                           dtype={'id': str})
    test_df['id'] = test_df['id'].str.zfill(6)

    ensemble_logits = None
    final_ids       = None

    for i, state in enumerate(all_states):
        print(f'\n  Model {i+1}/{len(all_states)} (seed={CFG["seeds"][i]})...')
        m = WideResNet(depth=CFG['depth'], widen=CFG['widen'],
                       dropout=0.0).to(DEVICE)
        m.load_state_dict(state)
        m.eval()

        logits, ids = predict_tta(m, test_df, DEVICE, n_views=CFG['tta_views'])

        if ensemble_logits is None:
            ensemble_logits = logits
            final_ids       = ids
        else:
            ensemble_logits = ensemble_logits + logits

        del m
        gc.collect()
        torch.cuda.empty_cache()

    # Average and decode
    ensemble_logits = ensemble_logits / len(all_states)
    final_preds     = ensemble_logits.argmax(dim=1).numpy()
    final_labels    = [I2C[p] for p in final_preds]

    # Write submission
    sub = pd.DataFrame({'id': final_ids, 'label': final_labels})
    # Ensure correct ordering matches sample_submission.csv
    sub = sub.set_index('id').reindex(test_df['id']).reset_index()
    sub.to_csv('submission.csv', index=False)

    dist = Counter(final_labels)
    print('\n  Prediction distribution:')
    for cls in CLASSES:
        print(f'    {cls:12s}: {dist.get(cls, 0):5d}')

    print(f'\n  submission.csv written ({len(sub)} rows)')
    print(f'  Seeds used: {CFG["seeds"]}')
    print(f'  TTA views:  {CFG["tta_views"]}')


if __name__ == '__main__':
    main()


# =============================================================================
# TIME BUDGET REFERENCE (single T4, Kaggle standard)
# =============================================================================
# WRN-28-10, batch=512, 1 seed:
#   ~36.5M params, ~1.3s/epoch on T4 with AMP
#   300 epochs × 1.3s = ~6.5h training
#   SWA BN update    = ~5 min
#   30-view TTA      = ~15 min
#   Total            = ~7h  ✓ fits in 9h limit
#
# 2-seed ensemble: reduce to 200 epochs per seed
#   200 × 1.3s × 2 = ~8.7h training + ~30 min inference = ~9.2h  (tight)
#   Safer: 180 epochs per seed = ~7.8h + 0.5h = ~8.3h  ✓
#
# If you want 2 seeds: change CFG to:
#   'epochs': 180, 'swa_start': 144, 'seeds': [42, 137]
# =============================================================================

Device: cuda
GPU: Tesla T4
Training samples: 27930
Class distribution:
  airplane    : 4750
  automobile  : 4750
  bird        : 4750
  cat         : 4750
  deer        : 3800
  dog         : 3800
  frog        : 475
  horse       : 475
  ship        : 285
  truck       : 95

  Training seed=42  |  epochs=300
  batch=512  lr=0.1  SWA from ep240
  WRN-28-10 | 36.5M params
  Ep   5/300 | loss 1.8994 | val F1 0.1673 | lr 0.10000
  -> New best F1 0.1673 saved to /kaggle/working/best_seed42.pth
  Ep  10/300 | loss 1.6973 | val F1 0.3452 | lr 0.09989
  -> New best F1 0.3452 saved to /kaggle/working/best_seed42.pth
  Ep  15/300 | loss 1.6032 | val F1 0.3859 | lr 0.09955
  -> New best F1 0.3859 saved to /kaggle/working/best_seed42.pth
  Ep  20/300 | loss 1.5482 | val F1 0.4130 | lr 0.09900
  -> New best F1 0.4130 saved to /kaggle/working/best_seed42.pth
  Ep  25/300 | loss 1.4792 | val F1 0.3812 | lr 0.09822
  Ep  30/300 | loss 1.4225 | val F1 0.5024 | lr 0.09723
  -> New best F1 0.5024 saved 